In [ ]:
import ijson
import json
import sqlite3
from pathlib import Path


HMDB_JSON_PATH = "../data/hmdb_metabolites_compact.json"
HMDB_ONTOLOGY_JSON_PATH = "../data/hmdb_metabolites_ontology.json"
SQLITE_PATH = "../data/hmdb.sqlite"


def to_json_text(x):
    if x is None:
        return None
    return json.dumps(x, ensure_ascii=False)


def safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def ensure_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def normalize_external_links(external_links):
    """
    Returns a flat dict of lowercase key -> value(s), defensively.
    Handles cases where values are strings, dicts, or lists.
    """
    out = {}

    if not isinstance(external_links, dict):
        return out

    for k, v in external_links.items():
        lk = str(k).strip().lower()
        out[lk] = v

    return out


def extract_external_id(external_links, candidate_keys):
    """
    Try multiple possible keys in external_links and return a best-effort string.
    If value is a list/dict, serialize or extract something sensible.
    """
    ext = normalize_external_links(external_links)

    for key in candidate_keys:
        if key not in ext:
            continue

        value = ext[key]

        if value is None:
            return None

        if isinstance(value, str):
            return value.strip() or None

        if isinstance(value, (int, float)):
            return str(value)

        if isinstance(value, list):
            # prefer first string-like item
            for item in value:
                if isinstance(item, str) and item.strip():
                    return item.strip()
                if isinstance(item, dict):
                    for subkey in ("id", "accession", "identifier", "value", "name"):
                        subval = item.get(subkey)
                        if isinstance(subval, str) and subval.strip():
                            return subval.strip()
            return to_json_text(value)

        if isinstance(value, dict):
            for subkey in ("id", "accession", "identifier", "value", "name"):
                subval = value.get(subkey)
                if isinstance(subval, str) and subval.strip():
                    return subval.strip()
            return to_json_text(value)

        return str(value)

    return None


def extract_metabolite_row(hmdb_id, obj):
    external_links = obj.get("external_links", {})

    foodb_id = extract_external_id(
        external_links,
        {"foodb", "foodb_id"}
    )
    kegg_id = extract_external_id(
        external_links,
        {"kegg", "kegg_id", "kegg_compound"}
    )
    pdb_id = extract_external_id(
        external_links,
        {"pdb", "pdb_id"}
    )
    chebi_id = extract_external_id(
        external_links,
        {"chebi", "chebi_id"}
    )

    row = (
        hmdb_id,
        obj.get("name"),
        obj.get("description"),
        obj.get("cas"),
        obj.get("moldb_inchi"),
        obj.get("moldb_inchikey"),
        obj.get("moldb_smiles"),
        to_json_text(obj.get("chemical_taxonomy")),
        to_json_text(obj.get("retention_indices")),
        to_json_text(obj.get("biospecimen_normal")),
        to_json_text(obj.get("biospecimen_abnormal")),
        foodb_id,
        kegg_id,
        pdb_id,
        chebi_id,
        to_json_text(obj),  # full raw object for future projects
    )
    return row


def extract_ontology_row(hmdb_id, obj):
    functional_ontology = obj.get("functional_ontology", {})

    effects = safe_get(
        functional_ontology,
        "health_effect",
        "physiological_effect",
        default=[]
    )
    physiological_effects = [
        f"{e.get('name', '<No name>')}: {e.get('description', '<No description>')} [{e.get('category', 'N/A')}]"
        for e in ensure_list(effects)
        if isinstance(e, dict)
    ]

    disposition_items = safe_get(
        functional_ontology,
        "disposition",
        "biological_location",
        default=[]
    )
    disposition = [
        d.get("name")
        for d in ensure_list(disposition_items)
        if isinstance(d, dict) and d.get("name")
    ]

    pr = safe_get(functional_ontology, "process", default=[])
    metabolic_processes = [
        f"{p.get('name', '<No name>')}: {p.get('description', '<No description>')} [{p.get('category', 'N/A')}]"
        for p in ensure_list(pr)
        if isinstance(p, dict) and p.get("process_type") == "Metabolic"
    ]

    diseases = [
        f"{p.get('name', '<No name>')}: {p.get('description', '<No description>')} [{p.get('category', 'N/A')}]"
        for p in ensure_list(pr)
        if isinstance(p, dict) and str(p.get("process_type", "")).lower() == "disease"
    ]

    roles = safe_get(functional_ontology, "role", default=[])
    biomarker_for = [
        f"{r.get('name', '<No name>')}: {r.get('description', '<No description>')}"
        for r in ensure_list(roles)
        if isinstance(r, dict) and "Biomarker" in str(r.get("category", ""))
    ]

    row = (
        hmdb_id,
        to_json_text(physiological_effects),
        to_json_text(disposition),
        to_json_text(metabolic_processes),
        to_json_text(diseases),
        to_json_text(biomarker_for),
        to_json_text(obj),  # full raw ontology object
    )
    return row


def create_tables(conn):
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS metabolites (
        hmdb_id TEXT PRIMARY KEY,
        name TEXT,
        description TEXT,
        cas TEXT,
        moldb_inchi TEXT,
        moldb_inchikey TEXT,
        moldb_smiles TEXT,
        chemical_taxonomy_json TEXT,
        retention_indices_json TEXT,
        biospecimen_normal_json TEXT,
        biospecimen_abnormal_json TEXT,
        foodb_id TEXT,
        kegg_id TEXT,
        pdb_id TEXT,
        chebi_id TEXT,
        raw_json TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS ontology (
        hmdb_id TEXT PRIMARY KEY,
        physiological_effects_json TEXT,
        disposition_json TEXT,
        metabolic_processes_json TEXT,
        diseases_json TEXT,
        biomarker_for_json TEXT,
        raw_json TEXT
    )
    """)

    cur.execute("CREATE INDEX IF NOT EXISTS idx_metabolites_name ON metabolites(name)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_metabolites_inchikey ON metabolites(moldb_inchikey)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_metabolites_inchi ON metabolites(moldb_inchi)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_metabolites_smiles ON metabolites(moldb_smiles)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_metabolites_cas ON metabolites(cas)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_metabolites_foodb_id ON metabolites(foodb_id)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_metabolites_kegg_id ON metabolites(kegg_id)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_metabolites_pdb_id ON metabolites(pdb_id)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_metabolites_chebi_id ON metabolites(chebi_id)")

    conn.commit()


def configure_sqlite_for_bulk_load(conn):
    cur = conn.cursor()
    cur.execute("PRAGMA journal_mode = WAL")
    cur.execute("PRAGMA synchronous = NORMAL")
    cur.execute("PRAGMA temp_store = MEMORY")
    cur.execute("PRAGMA cache_size = -200000")  # approx 200 MB cache
    conn.commit()


def load_metabolites_to_sqlite(conn, json_path, batch_size=1000):
    if not Path(json_path).exists():
        print(f"Metabolite file not found: {json_path}")
        return

    cur = conn.cursor()
    insert_sql = """
    INSERT OR REPLACE INTO metabolites (
        hmdb_id,
        name,
        description,
        cas,
        moldb_inchi,
        moldb_inchikey,
        moldb_smiles,
        chemical_taxonomy_json,
        retention_indices_json,
        biospecimen_normal_json,
        biospecimen_abnormal_json,
        foodb_id,
        kegg_id,
        pdb_id,
        chebi_id,
        raw_json
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """

    batch = []
    count = 0

    with open(json_path, "rb") as f:
        for hmdb_id, obj in ijson.kvitems(f, ""):
            try:
                batch.append(extract_metabolite_row(hmdb_id, obj))
                count += 1

                if len(batch) >= batch_size:
                    cur.executemany(insert_sql, batch)
                    conn.commit()
                    print(f"Inserted metabolites: {count}")
                    batch.clear()

            except Exception as e:
                print(f"failed metabolite {hmdb_id}: {e}")

    if batch:
        cur.executemany(insert_sql, batch)
        conn.commit()
        print(f"Inserted metabolites: {count}")


def load_ontology_to_sqlite(conn, json_path, batch_size=1000):
    if not Path(json_path).exists():
        print(f"Ontology file not found: {json_path}")
        return

    cur = conn.cursor()
    insert_sql = """
    INSERT OR REPLACE INTO ontology (
        hmdb_id,
        physiological_effects_json,
        disposition_json,
        metabolic_processes_json,
        diseases_json,
        biomarker_for_json,
        raw_json
    ) VALUES (?, ?, ?, ?, ?, ?, ?)
    """

    batch = []
    count = 0

    with open(json_path, "rb") as f:
        for hmdb_id, obj in ijson.kvitems(f, ""):
            try:
                batch.append(extract_ontology_row(hmdb_id, obj))
                count += 1

                if len(batch) >= batch_size:
                    cur.executemany(insert_sql, batch)
                    conn.commit()
                    print(f"Inserted ontology rows: {count}")
                    batch.clear()

            except Exception as e:
                print(f"failed ontology {hmdb_id}: {e}")

    if batch:
        cur.executemany(insert_sql, batch)
        conn.commit()
        print(f"Inserted ontology rows: {count}")


conn = sqlite3.connect(SQLITE_PATH)
configure_sqlite_for_bulk_load(conn)
create_tables(conn)

load_metabolites_to_sqlite(conn, HMDB_JSON_PATH, batch_size=1000)
load_ontology_to_sqlite(conn, HMDB_ONTOLOGY_JSON_PATH, batch_size=1000)

conn.close()

print(f"Done. SQLite DB saved to: {SQLITE_PATH}")

Inserted metabolites: 1000
Inserted metabolites: 2000
Inserted metabolites: 3000
Inserted metabolites: 4000
Inserted metabolites: 5000
Inserted metabolites: 6000
Inserted metabolites: 7000
Inserted metabolites: 8000
Inserted metabolites: 9000
Inserted metabolites: 10000
Inserted metabolites: 11000
Inserted metabolites: 12000
Inserted metabolites: 13000
Inserted metabolites: 14000
Inserted metabolites: 15000
Inserted metabolites: 16000
Inserted metabolites: 17000
Inserted metabolites: 18000
Inserted metabolites: 19000
Inserted metabolites: 20000
Inserted metabolites: 21000
Inserted metabolites: 22000
Inserted metabolites: 23000
Inserted metabolites: 24000
Inserted metabolites: 25000
Inserted metabolites: 26000
Inserted metabolites: 27000
Inserted metabolites: 28000
Inserted metabolites: 29000
Inserted metabolites: 30000
Inserted metabolites: 31000
Inserted metabolites: 32000
Inserted metabolites: 33000
Inserted metabolites: 34000
Inserted metabolites: 35000
Inserted metabolites: 36000
I

In [16]:
import sqlite3
import json

SQLITE_PATH = "../data/hmdb.sqlite"


def ensure_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def to_json_text(x):
    if x is None:
        return None
    return json.dumps(x, ensure_ascii=False)


conn = sqlite3.connect(SQLITE_PATH)
cur = conn.cursor()

cur.execute("""
SELECT hmdb_id, raw_json
FROM ontology
WHERE raw_json IS NOT NULL
""")

rows = cur.fetchall()

update_rows = []

for hmdb_id, raw_json_text in rows:
    try:
        obj = json.loads(raw_json_text)
        functional_ontology = obj.get("functional_ontology", {})

        if functional_ontology is None:
             continue

        pr = functional_ontology.get("process", [])

        metabolic_processes = [
            f"{p.get('name', '<No name>')}: {p.get('description', '<No description>')} [{p.get('category', 'N/A')}]"
            for p in ensure_list(pr)
            if isinstance(p, dict) and p.get("process_type") == "Metabolic"
        ]

        diseases = [
            f"{p.get('name', '<No name>')}: {p.get('description', '<No description>')} [{p.get('category', 'N/A')}]"
            for p in ensure_list(pr)
            if isinstance(p, dict) and str(p.get("process_type", "")).lower() == "disease"
        ]

        update_rows.append((
            to_json_text(metabolic_processes),
            to_json_text(diseases),
            hmdb_id
        ))

    except Exception as e:
        print(f"failed for {hmdb_id}: {e}")

cur.executemany("""
UPDATE ontology
SET metabolic_processes_json = ?,
    diseases_json = ?
WHERE hmdb_id = ?
""", update_rows)

conn.commit()
conn.close()

print(f"Updated {len(update_rows)} ontology rows.")

Updated 246971 ontology rows.


In [14]:
import sqlite3
import json

DB_PATH = "../data/hmdb.sqlite"

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# get rows that have raw_json
rows = cur.execute("""
SELECT hmdb_id, raw_json
FROM metabolites
WHERE raw_json IS NOT NULL
""").fetchall()

updates = []

for row in rows:
    hmdb_id = row["hmdb_id"]

    try:
        obj = json.loads(row["raw_json"])
    except Exception:
        print(f"failed to parse JSON for {hmdb_id}")
        continue

    syns = obj.get("synonyms", [])

    if not syns:
        synonyms_str = None
    else:
        names = [
            s.get("name")
            for s in syns
            if isinstance(s, dict) and s.get("name")
        ]

        synonyms_str = "; ".join(names) if names else None

    updates.append((synonyms_str, hmdb_id))


# batch update
cur.executemany("""
UPDATE metabolites
SET synonyms = ?
WHERE hmdb_id = ?
""", updates)

conn.commit()
conn.close()

print("Synonyms column populated.")

Synonyms column populated.


In [12]:
import ijson
import json

# HMDB_JSON_PATH = "../data/hmdb_metabolites_compact.json"
HMDB_JSON_PATH = "../data/hmdb_metabolites_ontology.json"

with open(HMDB_JSON_PATH, "rb") as f:
    for hmdb_id, entry in ijson.kvitems(f, ""):
        print("HMDB ID:", hmdb_id)
        print(json.dumps(entry, indent=2))  # truncate to avoid flooding
        break

HMDB ID: HMDB0000001
{
  "hmdb_id": "HMDB0000001",
  "status": "Detected and Quantified",
  "created_at": "2005-11-16T15:48:42Z",
  "updated_at": "2023-02-21T17:14:26Z",
  "name": "1-Methylhistidine",
  "description": "1-Methylhistidine, also known as 1-MHis, 1MH, tau-methylhistidine or tele-methylhistidine, belongs to the class of organic compounds known as histidine and derivatives. 1MH is also classified as a methylamino acid. Methylamino acids are primarily proteogenic amino acids (found in proteins) which have been methylated (in situ) on their side chains by various methyltransferase enzymes. Histidine can be methylated at either the N1 or N3 position of its imidazole ring, yielding the isomers 1-methylhistidine (1MH; also referred to as tau-methylhistidine, according to IUPAC) or 3-methylhistidine (3MH; pi-methylhistidine, according to IUPAC), respectively. There is considerable confusion with regard to the nomenclature of the methylated nitrogen atoms on the imidazole ring of h